In [1]:
import numpy as np

path = '../data/penguins.csv'

# NA becomes nan automatically for float columns
P = np.genfromtxt(path, delimiter=',', skip_header=1, usecols=(2, 3, 4, 5))
penguin_species = np.genfromtxt(path, delimiter=',', skip_header=1, usecols=0, dtype=str)

print("P shape:", P.shape)
print("penguin_species shape:", penguin_species.shape)

P shape: (344, 4)
penguin_species shape: (344,)


In [2]:
col_names = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

print("Total missing:", np.sum(np.isnan(P)))

miss_per_col = np.sum(np.isnan(P), axis=0)
for i in range(len(col_names)):
    print(col_names[i], ":", miss_per_col[i])

rows_with_nan = np.any(np.isnan(P), axis=1)
print("Rows with at least one missing value:", np.sum(rows_with_nan))
print(np.where(rows_with_nan)[0])
print(P[rows_with_nan])

print("P.mean(axis=0):", P.mean(axis=0))
print("Correct means (nanmean):", np.nanmean(P, axis=0))

Total missing: 8
bill_length_mm : 2
bill_depth_mm : 2
flipper_length_mm : 2
body_mass_g : 2
Rows with at least one missing value: 2
[  3 271]
[[nan nan nan nan]
 [nan nan nan nan]]
P.mean(axis=0): [nan nan nan nan]
Correct means (nanmean): [  43.92192982   17.15116959  200.91520468 4201.75438596]


Only 2 rows have missing values, and in both rows all four measurements are missing. They are completely empty rows, not partly filled ones. P.mean(axis=0) gives nan for every column, because a single nan makes the whole mean nan. np.nanmean ignores the nans and gives the correct means.

In [3]:
means = np.nanmean(P, axis=0)
medians = np.nanmedian(P, axis=0)

P_mean = P.copy()
P_median = P.copy()
for j in range(P.shape[1]):
    mask = np.isnan(P[:, j])
    P_mean[mask, j] = means[j]
    P_median[mask, j] = medians[j]

print("nan left in P_mean:", np.isnan(P_mean).any())
print("nan left in P_median:", np.isnan(P_median).any())

print("Mean fill values:", np.round(means, 3))
print("Median fill values:", np.round(medians, 3))
print("Difference:", np.round(np.abs(means - medians), 3))

nan left in P_mean: False
nan left in P_median: False
Mean fill values: [  43.922   17.151  200.915 4201.754]
Median fill values: [  44.45   17.3   197.   4050.  ]
Difference: [5.28000e-01 1.49000e-01 3.91500e+00 1.51754e+02]


Body mass has the biggest gap between mean and median (about 150 g). That's partly because its values are in the thousands, and partly because its distribution is right-skewed, with a few heavy penguins pulling the mean up.

Imputing these two rows isn't a good idea. They have no real measurements at all, so after imputation they would just become fake avg penguins that add no information and slightly reduce the variance. Since it's only 2 rows out of 344, dropping them is safer.

In [4]:
print("Min:", P_median.min(axis=0))
print("Max:", P_median.max(axis=0))

# min-max
mins = P_median.min(axis=0)
maxs = P_median.max(axis=0)
P_minmax = (P_median - mins) / (maxs - mins)
print("Min-max col mins:", P_minmax.min(axis=0))
print("Min-max col maxs:", P_minmax.max(axis=0))

# z-score
mu = P_median.mean(axis=0)
sigma = P_median.std(axis=0)
P_z = (P_median - mu) / sigma
print("Z-score mean ~0:", np.allclose(P_z.mean(axis=0), 0))
print("Z-score std ~1:", np.allclose(P_z.std(axis=0), 1))

Min: [  32.1   13.1  172.  2700. ]
Max: [  59.6   21.5  231.  6300. ]
Min-max col mins: [0. 0. 0. 0.]
Min-max col maxs: [1. 1. 1. 1.]
Z-score mean ~0: True
Z-score std ~1: True


Body mass would dominate the distance. Its values are around 2700 to 6300, while the other features are in the tens or low hundreds, so a difference of a few hundred grams outweighs everything else. That's why scaling is needed before distance-based methods like KNN.

In [5]:
values = np.array([1.0, 2.0, 3.0, 4.0, 100.0])

mm = (values - values.min()) / (values.max() - values.min())
zz = (values - values.mean()) / values.std()

print("Min-max:", np.round(mm, 3))
print("Z-score:", np.round(zz, 3))

Min-max: [0.   0.01 0.02 0.03 1.  ]
Z-score: [-0.538 -0.513 -0.487 -0.461  1.999]


With min-max, the four normal values get squashed into roughly 0 to 0.03 because the outlier takes the value 1, so you can barely tell them apart. With z-score they also bunch together (around -0.54 to -0.46), but they aren't forced into a tiny fixed range, and the outlier clearly stands out at about 2.

With extreme outliers I'd choose z-score. Min-max depends completely on the min and max, so one outlier squeezes everything else. Z-score is also affected by outliers through the mean and std, just less severely. A robust scaler using the median and IQR would be even better.

In [6]:
island = np.genfromtxt(path, delimiter=',', skip_header=1, usecols=1, dtype=str)
sex = np.genfromtxt(path, delimiter=',', skip_header=1, usecols=6, dtype=str)

# label encoding
sp_cats, sp_codes = np.unique(penguin_species, return_inverse=True)
print("Species categories:", sp_cats)
print("First 5 species codes:", sp_codes[:5])

is_cats, is_codes = np.unique(island, return_inverse=True)
print("Island categories:", is_cats)
print("First 5 island codes:", is_codes[:5])

# one-hot for species
species_onehot = (penguin_species.reshape(-1, 1) == sp_cats).astype(int)
print("One-hot shape:", species_onehot.shape)
print("Every row sums to 1:", np.all(species_onehot.sum(axis=1) == 1))
print("Column sums:", species_onehot.sum(axis=0))

# sex
sex_vals, sex_counts = np.unique(sex, return_counts=True)
for v, c in zip(sex_vals, sex_counts):
    print(v, ":", c)

# final feature matrix
X = np.hstack([P_z, species_onehot])
print("Feature matrix shape:", X.shape)

Species categories: ['Adelie' 'Chinstrap' 'Gentoo']
First 5 species codes: [0 0 0 0 0]
Island categories: ['Biscoe' 'Dream' 'Torgersen']
First 5 island codes: [2 2 2 2 2]
One-hot shape: (344, 3)
Every row sums to 1: True
Column sums: [152  68 124]
NA : 11
female : 165
male : 168
Feature matrix shape: (344, 7)


The column sums of the one-hot array are the number of penguins of each species (Adelie, Chinstrap, Gentoo). They add up to 344.

Sex has 2 categories: 165 female and 168 male, plus 11 NA. To handle the NA values, you could fill them with the most frequent category, or drop those rows since there are only 11. A better option is to predict sex from body mass and bill size, because males are bigger. Treating NA as its own category is also possible but less meaningful here.

The final matrix X has shape (344, 7): 4 scaled measurements plus 3 species columns.